In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/11000
1


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

interpolate adjoint :  True True True
RUN  0 , total integrated cost =  38727.35641379273
Gradient descend method:  None
RUN  0 , total integrated cost =  38727.35641379273
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23532.636143093983
Gradient descend method:  None
RUN  0 , total integrated cost =  23532.636143093983
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  140 0.4500000000000001 0.9000000000000006
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10019.968518582271
Gradient descend method:  None
RUN  0 , total integrated cost =  10019.968518582271
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN 

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  33.637592881489994
RUN  2 , total integrated cost =  25.5713838467463
RUN  3 , total integrated cost =  13.646444415817509
RUN  4 , total integrated cost =  11.650913497940916
RUN  5 , total integrated cost =  9.03651098918099
RUN  6 , total integrated cost =  7.267164126831273
RUN  7 , total integrated cost =  5.099761269404373
RUN  8 , total integrated cost =  4.798353358602128
RUN  9 , total integrated cost =  4.494462935277617
RUN  10 , total integrated cost =  3.6649959417845817
RUN  11 , total integrated cost =  3.2550927233937585
RUN  12 , total integrated cost =  3.1410269280271987
RUN  13 , total integrated cost =  3.0244958825618906
RUN  14 , total integrated cost =  3.010865177107536
RUN  15 , total integrated cost =  2.972902903768891
RUN  16

ERROR:root:Problem in initial value trasfer


RUN  40 , total integrated cost =  2.290317583645329
Control only changes marginally.
RUN  42 , total integrated cost =  2.2903175836453222
Improved over  42  iterations in  13.940880408510566  seconds by  99.95506793490583  percent.
Problem in initial value trasfer:  Vmean_exc -67.8048355480298 -67.808635079433
weight =  22255.82104682076
set cost params:  1.0 22255.82104682076 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4999.779333522038
Gradient descend method:  None
RUN  1 , total integrated cost =  4334.126758746933
RUN  2 , total integrated cost =  4331.924840113133
RUN  3 , total integrated cost =  4331.903849079088
RUN  4 , total integrated cost =  4331.901207550202
RUN  5 , total integrated cost =  4331.900615497839
RUN  6 , total integrated cost =  4331.9004713400855
RUN  7 , total integrated cost =  4331.900363964012
RUN  8 , total integrated cost =  4331.900301956359
RUN  9 , total integrated cost =  4331.900301956346
RUN  10 , total integrat

ERROR:root:Problem in initial value trasfer


RUN  11 , total integrated cost =  4331.900301956345
Control only changes marginally.
RUN  11 , total integrated cost =  4331.900301956345
Improved over  11  iterations in  0.333049263805151  seconds by  13.35817017138659  percent.
Problem in initial value trasfer:  Vmean_exc -61.163560379256694 -61.20366733312479
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  247.16191308801922
RUN  2 , total integrated cost =  175.6451040449948
RUN  3 , total integrated cost =  114.36040677612303
RUN  4 , total integrated cost =  93.5443619667394
RUN  5 , total integrated cost =  78.92325910801308
RUN  6 , total integrated cost =  67.55856687024362
RUN  7 , total integrated cost =  61.445710064576545
RUN  8 , total integrated cost =  51.52544273733108
RUN  9 , total integrated cost =  48.776300200022845
RUN  10 , 

...

RUN  6 , total integrated cost =  14176.406243447615
RUN  7 , total integrated cost =  14176.406243447615
Control only changes marginally.
RUN  7 , total integrated cost =  14176.406243447615
Improved over  7  iterations in  0.4425628539174795  seconds by  35.21598085131808  percent.
Problem in initial value trasfer:  Vmean_exc -56.66327500944693 -56.66556221957359
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  779.030108635771
RUN  2 , total integrated cost =  506.39189297273145
RUN  3 , total integrated cost =  338.80938203923114
RUN  4 , total integrated cost =  282.09791028560926
RUN  5 , total integrated cost =  240.90174496525498
RUN  6 , total integrated cost =  203.64463351107298
RUN  7 , total integrated cost =  189.06860030434896
RUN  8 , total integrated cost =  185.3761291356274
RUN  9 ,

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  53 , total integrated cost =  156.4542068505888
Improved over  53  iterations in  2.098528891801834  seconds by  99.53002714037179  percent.
Problem in initial value trasfer:  Vmean_exc -62.0491308376085 -62.061218830980934
weight =  2127.7824442694064
set cost params:  1.0 2127.7824442694064 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  30540.190155161894
Gradient descend method:  None
RUN  1 , total integrated cost =  29618.46669797053
RUN  2 , total integrated cost =  20499.60463499186
RUN  3 , total integrated cost =  20019.867926529027
RUN  4 , total integrated cost =  19920.742257293703
RUN  5 , total integrated cost =  19920.7422572937


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  19920.7422572937
Control only changes marginally.
RUN  6 , total integrated cost =  19920.7422572937
Improved over  6  iterations in  0.2351425737142563  seconds by  34.77204249192698  percent.
Problem in initial value trasfer:  Vmean_exc -56.68745566232517 -56.68990472511955


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
[0, 5] []
closest index  5
set 

ERROR:root:Problem in initial value trasfer


RUN  90 , total integrated cost =  13.046655082010506
Control only changes marginally.
RUN  92 , total integrated cost =  13.046655082010501
Improved over  92  iterations in  3.5559059027582407  seconds by  99.85540369373393  percent.
Problem in initial value trasfer:  Vmean_exc -67.32056202792239 -67.3303687948532
weight =  6983.749039839579
set cost params:  1.0 6983.749039839579 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8885.005109065776
Gradient descend method:  None
RUN  1 , total integrated cost =  7852.426877336119
RUN  2 , total integrated cost =  7842.93158404723
RUN  3 , total integrated cost =  7842.92258356871
RUN  4 , total integrated cost =  7842.921365572161
RUN  5 , total integrated cost =  7842.921016525292
RUN  6 , total integrated cost =  7842.920915578565
RUN  7 , total integrated cost =  7842.920864028497
RUN  8 , total integrated cost =  7842.9208612690745
RUN  9 , total integrated cost =  7842.920858401466
RUN  10 , total integra

ERROR:root:Problem in initial value trasfer


RUN  11 , total integrated cost =  7842.920858401461
Control only changes marginally.
RUN  11 , total integrated cost =  7842.920858401461
Improved over  11  iterations in  0.5502340886741877  seconds by  11.728572329137194  percent.
Problem in initial value trasfer:  Vmean_exc -59.23414523154368 -59.25291955653843
-------  15 0.4500000000000001 0.4500000000000002
found solution for  15
-------  20 0.4500000000000001 0.4750000000000002
[0, 5, 15] []
closest index  15
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  51.639407453488495
Gradient descend method:  None
RUN  1 , total integrated cost =  28.376083627475005
RUN  2 , total integrated cost =  28.376039923793297
RUN  3 , total integrated cost =  28.376014817281792
RUN  4 , total integrated cost =  28.3760070406713
RUN  5 , total integrated cost =  28.37599996480459
RUN  6 , total integrated cost =  28.375996943349836
RUN  7 , total integrated cost =  28.3

...

RUN  7 , total integrated cost =  23357.509804620844
Control only changes marginally.
RUN  7 , total integrated cost =  23357.509804620844
Improved over  7  iterations in  0.47118863835930824  seconds by  34.4998255329928  percent.
Problem in initial value trasfer:  Vmean_exc -56.69613097868317 -56.698125863240286
-------  135 0.5250000000000001 0.8750000000000006
-------  140 0.4500000000000001 0.9000000000000006
-------  145 0.5750000000000002 0.9000000000000006
[0, 5, 15, 25, 30, 70, 85, 100, 110, 115, 125, 140, 10, 20, 50, 55, 135, 65, 95, 45] [125, 110, 140, 135, 115, 100, 85, 95, 70, 65, 55, 50, 45, 30, 25, 20, 15, 10, 5]
closest index  0
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32994.6380471684
Gradient descend method:  None
RUN  1 , total integrated cost =  773.9361011474083
RUN  2 , total integrated cost =  507.6353092359736
RUN  3 , total integrated cost =  339.256535419018
RUN  4 , total inte

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  65 , total integrated cost =  153.84444118680912
Improved over  65  iterations in  2.554541740566492  seconds by  99.5337289623639  percent.
Problem in initial value trasfer:  Vmean_exc -62.13731583568501 -62.149878646475244
weight =  2163.877434248958
set cost params:  1.0 2163.877434248958 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  30818.728407577924
Gradient descend method:  None
RUN  1 , total integrated cost =  30205.3448896072
RUN  2 , total integrated cost =  20500.596147122407
RUN  3 , total integrated cost =  20161.070791640137
RUN  4 , total integrated cost =  20084.85943603875


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  20084.85943603874
RUN  6 , total integrated cost =  20084.85943603874
Control only changes marginally.
RUN  6 , total integrated cost =  20084.85943603874
Improved over  6  iterations in  0.3957982826977968  seconds by  34.82904560364621  percent.
Problem in initial value trasfer:  Vmean_exc -56.68830615918797 -56.69067439850909
------------------------------------------------------------
-------------------- 20
------------------------------------------------------------
found solution:  [0, 5, 15, 25, 30, 70, 85, 100, 110, 115, 125, 140, 10, 20, 50, 55, 135, 65, 95, 45]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.550000

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  26187.130458348674
set cost params:  1.0 26187.130458348674 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5074.961766653717
Gradient descend method:  None
RUN  1 , total integrated cost =  4993.257205194387
RUN  2 , total integrated cost =  4356.451633514245
RUN  3 , total integrated cost =  4355.375108386065
RUN  4 , total integrated cost =  435

ERROR:root:Problem in initial value trasfer


RUN  11 , total integrated cost =  4355.350388453147
RUN  12 , total integrated cost =  4355.350388453145
RUN  13 , total integrated cost =  4355.350388453145
Control only changes marginally.
RUN  13 , total integrated cost =  4355.350388453145
Improved over  13  iterations in  0.5854301545768976  seconds by  14.17964137048196  percent.
Problem in initial value trasfer:  Vmean_exc -56.63155477276472 -56.63142972999077
-------  10 0.4250000000000001 0.42500000000000016
no convergence
weight =  8112.319854156981
set cost params:  1.0 8112.319854156981 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9084.498335462642
Gradient descend method:  None
RUN  1 , total integrated cost =  8554.351433591322
RUN  2 , total integrated cost =  6864.136363578484
RUN  3 , total integrated cost =  6834.409173371502
RUN  4 , total integrated cost =  6828.911747684625
RUN  5 , total integrated cost =  6825.381738454926
RUN  6 

ERROR:root:Problem in initial value trasfer


, total integrated cost =  6825.38173845492
RUN  7 , total integrated cost =  6825.38173845492
Control only changes marginally.
RUN  7 , total integrated cost =  6825.38173845492
Improved over  7  iterations in  0.3999219611287117  seconds by  24.86781893270799  percent.
Problem in initial value trasfer:  Vmean_exc -56.626766135338926 -56.62698722950579
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  5193.36965804162
set cost params:  1.0 5193.36965804162 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12968.994595970951
Gradient descend method:  None
RUN  1 , total integrated cost =  12966.450893533518
RUN  2 , total integrated cost =  12966.449892028764
RUN  3 , total integrated cost =  12966.448619766368
RUN  4 , total integrated cost =  12966.447937267494
RUN  5 , total integrated cost =  12966.447447278375
RUN  6 , total integrated cost =  12966.445778080604
RUN  7 , total integrated cost =  12966.437151730785
RUN  8 , total i

...

RUN  2 , total integrated cost =  33920.99582063309
RUN  3 , total integrated cost =  33920.99582063309
Control only changes marginally.
RUN  3 , total integrated cost =  33920.99582063309
Improved over  3  iterations in  0.22330877743661404  seconds by  0.29240383641989354  percent.
Problem in initial value trasfer:  Vmean_exc -56.7037157140787 -56.7035286032759
-------  135 0.5250000000000001 0.8750000000000006
no convergence
weight =  9354.525649445737
set cost params:  1.0 9354.525649445737 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  20494.711508789158
Gradient descend method:  None
RUN  1 , total integrated cost =  20404.423281043753
RUN  2 , total integrated cost =  20404.297440340146
RUN  3 , total integrated cost =  20404.29735377118
RUN  4 , total integrated cost =  20404.297353771162
RUN  5 , total integrated cost =  20404.29735377116


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  20404.29735377116
Control only changes marginally.
RUN  6 , total integrated cost =  20404.29735377116
Improved over  6  iterations in  0.3826885484158993  seconds by  0.4411584665596564  percent.
Problem in initial value trasfer:  Vmean_exc -56.69420409318475 -56.69470651976865
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
no convergence
weight =  8652.157350566153
set cost params:  1.0 8652.157350566153 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  29259.62007068663
Gradient descend method:  None
RUN  1 , total integrated cost =  29174.683696696524
RUN  2 , total integrated cost =  29174.68369669652


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  29174.68369669652
Control only changes marginally.
RUN  3 , total integrated cost =  29174.68369669652
Improved over  3  iterations in  0.2836009319871664  seconds by  0.29028529346901166  percent.
Problem in initial value trasfer:  Vmean_exc -56.70399695616619 -56.70408841926647


In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [False, False], [False, False], [False, False], [False, False], [True, True], [True, True], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [False, False], [False, False], [False, False], [False, False], [True, False], [True, True], [False, False], [True, True], [False, False], [False, False], [False, False], [False, False], [False, False], [True, True], [False, False]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4200.168485259583
Gradient descend method:  None
RUN  1 , total integrated cost =  25.1281957340964
RUN  2 , total integrated cost =  22.598142628052337
RUN  3 , total integrated cost =  19.448884897414025
RUN  4 , total integrated cost =  17.946619339523373
RUN  5 , total integrated cost =  16.252502873562122
RUN  6 , total integrated cost =  15.315785714411957
RUN  7 , total integrated cost =  14.225131441656151
RUN  8 , total integrated cost =  13.501188184232563
RUN  9 , total integrated cost =  12.427669126915918
RUN  10 , total integrated cost =  11.555922596053028
RUN  11 , total integrated cost =  9.973316826607482
RUN  12 , total integrated cost =  8.21632414573899
RUN  13 , total integrated cost =  7.727000230700886
RUN  14 , total integrated cost =  7.708424411676927
RUN  15 , total integrated cost =  7.667300576800366
RUN  

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  55 , total integrated cost =  6.726691355605357
Improved over  55  iterations in  10.882192073389888  seconds by  99.83984710662888  percent.
Problem in initial value trasfer:  Vmean_exc -67.7791837017595 -67.783251173926
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  22.27129162338579
Gradient descend method:  HS
RUN  1 , total integrated cost =  21.811246079221064
RUN  2 , total integrated cost =  21.600432158185235
RUN  3 , total integrated cost =  21.41037014807838
RUN  4 , total integrated cost =  21.377849445653048
RUN  5 , total integrated cost =  21.37665098388869
RUN  6 , total integrated cost =  21.376601231343816
RUN  7 , total integrated cost =  21.376601231343752
RUN  8 , total integrated cost =  21.376601231343745
RUN  9 , total integrated cost =  21.37660123134373


ERROR:root:Problem in initial value trasfer


RUN  10 , total integrated cost =  21.37660123134373
Control only changes marginally.
RUN  10 , total integrated cost =  21.37660123134373
Improved over  10  iterations in  1.6114385966211557  seconds by  4.017236212302095  percent.
Problem in initial value trasfer:  Vmean_exc -67.6369882144299 -67.64327603646502
weight =  23844.18368020896
set cost params:  1.0 23844.18368020896 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5020.298510042704
Gradient descend method:  None
RUN  1 , total integrated cost =  4622.626066524119
RUN  2 , total integrated cost =  4622.408442106399
RUN  3 , total integrated cost =  4622.393307941266
RUN  4 , total integrated cost =  4622.389480945705
RUN  5 , total integrated cost =  4622.388638830537
RUN  6 , total integrated cost =  4622.3883378938735
RUN  7 , total integrated cost =  4622.388169959392
RUN  8 , total integrated cost =  4622.387682980043
RUN  9 , total integrated cost =  4622.387472925729
RUN  10 , total integra

...

Control only changes marginally.
RUN  34 , total integrated cost =  57.816482714345526
Improved over  34  iterations in  4.024845020845532  seconds by  99.77266590665982  percent.
Problem in initial value trasfer:  Vmean_exc -64.40732707387662 -64.42369931478817
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1551.9305969320503
Gradient descend method:  HS
RUN  1 , total integrated cost =  1353.9216594792308
RUN  2 , total integrated cost =  1328.3078287426238
RUN  3 , total integrated cost =  1327.6811544343127
RUN  4 , total integrated cost =  1327.668118768803
RUN  5 , total integrated cost =  1326.7286733052076
RUN  6 , total integrated cost =  1326.7165207939183
RUN  7 , total integrated cost =  1326.7165139073604
RUN  8 , total integrated cost =  1326.7164858555302
RUN  9 , total integrated cost =  1326.7164840696275
RUN  10 , total integrated cost =  1326.7151066182914
RUN  11 , total integrated cost =  1326.7

ERROR:root:Problem in initial value trasfer


RUN  16 , total integrated cost =  1326.5098450188889
Control only changes marginally.
RUN  16 , total integrated cost =  1326.5098450188889
Improved over  16  iterations in  2.349883943796158  seconds by  14.525182528058068  percent.
Problem in initial value trasfer:  Vmean_exc -58.983823057310616 -58.97124175495139
weight =  2508.5970144423377
set cost params:  1.0 2508.5970144423377 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  31904.359938028105
Gradient descend method:  None
RUN  1 , total integrated cost =  30669.71317989274
RUN  2 , total integrated cost =  21751.452106974946
RUN  3 , total integrated cost =  21515.054206490116
RUN  4 , total integrated cost =  21482.542050424086
RUN  5 , total integrated cost =  21482.542050424076


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  21482.54205042407
RUN  7 , total integrated cost =  21482.54205042407
Control only changes marginally.
RUN  7 , total integrated cost =  21482.54205042407
Improved over  7  iterations in  0.6509103551506996  seconds by  32.66581090436435  percent.
Problem in initial value trasfer:  Vmean_exc -56.694346892870584 -56.69600698564929


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  26292.943265765723
set cost params:  1.0 26292.943265765723 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5085.937559522033
Gradient descend method:  None
RUN  1 , total integrated cost =  5085.7226238750945
RUN  2 , total integrated cost =  5085.704999717827
RUN  3 , total integrated cost =  5085.699053268408
RUN  4 , total integrated cost =  5085.696464503388
RUN  5 , total integrated cost =  5085.693723942542
RUN  6 , total i

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  25 , total integrated cost =  5074.85481042624
Improved over  25  iterations in  3.676986441016197  seconds by  0.2179096570905159  percent.
Problem in initial value trasfer:  Vmean_exc -61.279943029560435 -61.320372311454605
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  7027.0101024992555
set cost params:  1.0 7027.0101024992555 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10287.275749318233
Gradient descend method:  None
RUN  1 , total integrated cost =  10028.530327046785
RUN  2 , total integrated cost =  10028.527978009592
RUN  3 , total integrated cost =  10028.527978009586


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  10028.527978009586
Control only changes marginally.
RUN  4 , total integrated cost =  10028.527978009586
Improved over  4  iterations in  0.5322889536619186  seconds by  2.515221499003701  percent.
Problem in initial value trasfer:  Vmean_exc -56.647213355133 -56.647914292118664
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  10529.74211805325
set cost params:  1.0 10529.74211805325 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8229.316880575858
Gradient descend method:  None
RUN  1 , total integrated cost =  8101.3124228096085
RUN  2 , total integrated cost =  6466.858540268586
RUN  3 , total integrated cost =  6443.989198057112
RUN  4 , total integrated cost =  6421.138807135008
RUN  5 , total integrated cost =  6421.137974678921
RUN  6 , total integrated cost =  6421.137974678919


...

RUN  5 , total integrated cost =  23899.95295623309
Control only changes marginally.
RUN  5 , total integrated cost =  23899.95295623309
Improved over  5  iterations in  1.2709889356046915  seconds by  8.71464772131958e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.701114297050104 -56.70113790390735
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
no convergence
weight =  141043.77648818874
set cost params:  1.0 141043.77648818874 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23309.922977014714
Gradient descend method:  None
RUN  1 , total integrated cost =  23309.902844521726
RUN  2 , total integrated cost =  23309.90284452172
RUN  3 , total integrated cost =  23309.90284452171


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  23309.90284452171
Control only changes marginally.
RUN  4 , total integrated cost =  23309.90284452171
Improved over  4  iterations in  0.9328523278236389  seconds by  8.63687667447266e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.700342519729496 -56.700366395304805
-------  145 0.5750000000000002 0.9000000000000006
no convergence
weight =  121013.28759820254
set cost params:  1.0 121013.28759820254 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32974.165950558956
Gradient descend method:  None
RUN  1 , total integrated cost =  32974.13563609642
RUN  2 , total integrated cost =  32974.13563609639
RUN  3 , total integrated cost =  32974.135636096384


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  32974.135636096384
Control only changes marginally.
RUN  4 , total integrated cost =  32974.135636096384
Improved over  4  iterations in  0.9303977116942406  seconds by  9.193397831097627e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.70368217124903 -56.70366929093238
--------------- 101


In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  121.2621368284062
Gradient descend method:  None
RUN  1 , total integrated cost =  0.23020800000478142
RUN  2 , total integrated cost =  0.2297751169007645
RUN  3 , total integrated cost =  0.2297741912356492
RUN  4 , total integrated cost =  0.2297741843628024
RUN  5 , total integrated cost =  0.2297741842352678
RUN  6 , total integrated cost =  0.22977418423104592
RUN  7 , total integrated cost =  0.22977418423100074
RUN  8 , total integrated cost =  0.22977418423099769
RUN  9 , total integrated cost =  0.22977418423099763
RUN  10 , total integrated cost =  0.2297741842309976
RUN  11 , total integrated cost =  0.22977418423099732
RUN  12 , total integrated cost =  0.22977418423099724


ERROR:root:Problem in initial value trasfer


RUN  13 , total integrated cost =  0.22977418423099724
Control only changes marginally.
RUN  13 , total integrated cost =  0.22977418423099724
Improved over  13  iterations in  0.5615164637565613  seconds by  99.81051448519652  percent.
Problem in initial value trasfer:  Vmean_exc -67.92339377487903 -67.92625787835456
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12773.34920911489
Gradient descend method:  None
RUN  1 , total integrated cost =  46.54965537733865
RUN  2 , total integrated cost =  35.08466251242803
RUN  3 , total integrated cost =  22.819925032226447
RUN  4 , total integrated cost =  18.469376212853604
RUN  5 , total integrated cost =  14.534343331065447
RUN  6 , total integrated cost =  12.489607475244462
RUN  7 , total integrated cost =  10.715413334160436
RUN  8 , total integrated cost =  9.294466954610753
RUN  9 , total integrated cost =  8.054054633662611
RUN  1

ERROR:root:Problem in initial value trasfer


RUN  60 , total integrated cost =  3.0874397461276986
Control only changes marginally.
RUN  63 , total integrated cost =  3.0874397461276946
Improved over  63  iterations in  2.48744454421103  seconds by  99.97582905081836  percent.
Problem in initial value trasfer:  Vmean_exc -67.21763088296858 -67.22659594741512
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8077.815317409407
Gradient descend method:  None
RUN  1 , total integrated cost =  18.47353222758612
RUN  2 , total integrated cost =  14.25532716748371
RUN  3 , total integrated cost =  7.574186079983464
RUN  4 , total integrated cost =  5.881763806707499
RUN  5 , total integrated cost =  3.690578344364277
RUN  6 , total integrated cost =  3.2093240415166373
RUN  7 , total integrated cost =  2.7335524500969726
RUN  8 , total integrated cost =  2.1360582029291844
RUN  9 , total integrated cost =  1.8487112977049684
RUN  10 , t

...

RUN  6 , total integrated cost =  3.3078962189831778
RUN  7 , total integrated cost =  3.307896218983176
RUN  8 , total integrated cost =  3.3078962189831724
RUN  9 , total integrated cost =  3.3078962189831724
Control only changes marginally.
RUN  9 , total integrated cost =  3.3078962189831724
Improved over  9  iterations in  0.4351022858172655  seconds by  98.0195796732487  percent.
Problem in initial value trasfer:  Vmean_exc -71.69591059781801 -71.72737790212136
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23087.769447558632
Gradient descend method:  None
RUN  1 , total integrated cost =  110.30425192502994
RUN  2 , total integrated cost =  77.76210332178577
RUN  3 , total integrated cost =  46.097464503819644
RUN  4 , total integrated cost =  37.23256911313272
RUN  5 , total integrated cost =  29.259553106470744
RUN  6 , total integrated cost =  25.286405426283974
RUN  7 , 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  77 , total integrated cost =  9.374906445007568
Improved over  77  iterations in  3.1042737998068333  seconds by  99.95939449037594  percent.
Problem in initial value trasfer:  Vmean_exc -67.76261219447703 -67.79010191893816
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32660.065921829962
Gradient descend method:  None
RUN  1 , total integrated cost =  169.40661278225022
RUN  2 , total integrated cost =  74.67299603491023
RUN  3 , total integrated cost =  22.224697988404984
RUN  4 , total integrated cost =  21.781035039489907
RUN  5 , total integrated cost =  21.095872309959265
RUN  6 , total integrated cost =  20.675064925474878
RUN  7 , total integrated cost =  19.29337917745318
RUN  8 , total integrated cost =  19.231606868515595
RUN  9 , total integrated cost =  19.2109294494251
RUN  10 , total integrated cost =  18.92829567827299
RUN  11 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  39 , total integrated cost =  17.096347724476956
Improved over  39  iterations in  1.5753016788512468  seconds by  99.94765366436982  percent.
Problem in initial value trasfer:  Vmean_exc -64.81602270933874 -64.83193404524738


In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

ERROR:root:Problem in initial value trasfer


--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.22977418423099724
Gradient descend method:  None
RUN  1 , total integrated cost =  0.22977418423099724
Control only changes marginally.
RUN  1 , total integrated cost =  0.22977418423099724
Improved over  1  iterations in  0.11071484908461571  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.92339377487903 -67.92625787835456
-------  15 0.4

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  3.0874397461276946
Control only changes marginally.
RUN  1 , total integrated cost =  3.0874397461276946
Improved over  1  iterations in  0.11306323669850826  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.21763088296858 -67.22659594741512
-------  25 0.4250000000000001 0.5000000000000002
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.8937213486377189
Gradient descend method:  None
RUN  1 , total integrated cost =  0.8937213486377189
Control only changes marginally.
RUN  1 , total integrated cost =  0.8937213486377189
Improved over  1  iterations in  0.1128661185503006  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -70.96188194386252 -70.98184331940524
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7.19644577360

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


...

RUN  1 , total integrated cost =  10.047106894873366
Control only changes marginally.
RUN  1 , total integrated cost =  10.047106894873366
Improved over  1  iterations in  0.11656210198998451  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.06798057317918 -67.09292700279858
-------  115 0.4250000000000001 0.8250000000000005
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.26049445294910223
Gradient descend method:  None
RUN  1 , total integrated cost =  0.26049445294910223
Control only changes marginally.
RUN  1 , total integrated cost =  0.26049445294910223
Improved over  1  iterations in  0.11122235655784607  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -75.53609962929693 -75.57157010275938
-------  125 0.47500000000000014 0.8500000000000005
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  3.3078

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  3.3078962189831724
Control only changes marginally.
RUN  1 , total integrated cost =  3.3078962189831724
Improved over  1  iterations in  0.11375588923692703  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -71.69591059781801 -71.72737790212136
-------  135 0.5250000000000001 0.8750000000000006
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9.374906445007568
Gradient descend method:  None
RUN  1 , total integrated cost =  9.374906445007568
Control only changes marginally.
RUN  1 , total integrated cost =  9.374906445007568
Improved over  1  iterations in  0.11421710811555386  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.76261219447703 -67.79010191893816
-------  145 0.5750000000000002 0.9000000000000006
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  17.0963477244

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  17.096347724476956
Control only changes marginally.
RUN  1 , total integrated cost =  17.096347724476956
Improved over  1  iterations in  0.11606831476092339  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -64.81602270933874 -64.83193404524738
--------------- 12
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
